In [57]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import plotly.express as px #visualization
import plotly.graph_objects as go
import os
from imblearn.over_sampling import SMOTE

In [58]:
# Load the dataset
data = pd.read_csv("dataset/diagnosed_cbc_data_v4.csv")

# Check for missing values
missing_values = data.isnull().sum()
print("Missing Values:\n", missing_values)

Missing Values:
 WBC          0
LYMp         0
NEUTp        0
LYMn         0
NEUTn        0
RBC          0
HGB          0
HCT          0
MCV          0
MCH          0
MCHC         0
PLT          0
PDW          0
PCT          0
Diagnosis    0
dtype: int64


In [59]:
# Select numerical features for scaling
numerical_features = ['WBC', 'LYMp', 'NEUTp', 'LYMn', 'NEUTn', 'RBC', 'HGB', 'HCT', 'MCV', 'MCH', 'MCHC', 'PLT', 'PDW', 'PCT']

# Scale numerical features
scaler = StandardScaler()
data[numerical_features] = scaler.fit_transform(data[numerical_features])

In [60]:
data

,WBC,LYMp,NEUTp,LYMn,NEUTn,RBC,HGB,HCT,MCV,MCH,MCHC,PLT,PDW,PCT,Diagnosis
0,0.599843,2.466607e+00,-0.185600,1.811937,-0.049088,-0.688280,-1.281561,-0.209381,0.070161,-0.052056,-0.496853,-0.440741,-6.033850e-01,-0.131780,Normocytic hypochromic anemia
1,0.599843,2.352906e+00,-0.170704,1.737040,0.055399,-0.663423,-1.281561,-0.201751,0.088566,-0.057455,-3.497704,-0.537533,-6.033850e-01,-0.146376,Normocytic hypochromic anemia
2,-0.185996,6.900247e-01,-0.113827,0.239101,-0.258062,-0.262159,-0.835531,-0.149293,-0.323698,-0.085351,-0.678723,-0.881682,-4.165102e-03,-0.175570,Iron deficiency anemia
3,-0.522784,6.189614e-01,-0.094869,-0.060487,-0.467036,-0.173384,-2.199856,-0.127355,-0.290570,-0.079952,-0.587788,-0.935455,-1.002865e+00,-0.204763,Iron deficiency anemia
4,-1.027965,1.883888e+00,-0.161224,-0.210281,-0.989470,-0.276363,-3.091915,2.573771,-0.191184,-0.073653,-0.618099,0.064728,-5.035150e-01,-0.058796,Normocytic hypochromic anemia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1276,-0.971834,5.049352e-16,0.000000,0.000000,0.000000,0.053881,0.345135,0.000000,-0.187504,-0.039458,0.806547,-0.537533,-9.147597e-10,0.000000,Healthy
1277,-0.635046,5.049352e-16,0.000000,0.000000,0.000000,0.050330,0.738690,0.000000,0.217399,-0.009762,0.624678,-0.161120,-9.147597e-10,0.000000,Healthy
1278,0.375317,5.049352e-16,0.000000,0.000000,0.000000,-0.084609,0.240186,0.000000,0.106971,-0.025060,0.382184,1.064910,-9.147597e-10,0.000000,Healthy
1279,-0.388068,5.049352e-16,0.000000,0.000000,0.000000,0.014819,0.266424,0.000000,0.033352,-0.037658,0.109380,-0.602061,-9.147597e-10,0.000000,Healthy


In [61]:
# Separate features and target variable
X = data.drop(columns=['Diagnosis'])
y = data['Diagnosis']

# Encode target variable
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [62]:
# Correlation matrix
correlation_matrix = X.corr().abs()

# Select upper triangle of correlation matrix
upper_triangle = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))

# Find features with correlation greater than 0.9
highly_correlated_features = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.9)]

# Drop highly correlated features
X.drop(highly_correlated_features, axis=1, inplace=True)


In [63]:
X

,WBC,LYMp,NEUTp,LYMn,NEUTn,RBC,HGB,HCT,MCV,MCH,MCHC,PLT,PDW,PCT
0,0.599843,2.466607e+00,-0.185600,1.811937,-0.049088,-0.688280,-1.281561,-0.209381,0.070161,-0.052056,-0.496853,-0.440741,-6.033850e-01,-0.131780
1,0.599843,2.352906e+00,-0.170704,1.737040,0.055399,-0.663423,-1.281561,-0.201751,0.088566,-0.057455,-3.497704,-0.537533,-6.033850e-01,-0.146376
2,-0.185996,6.900247e-01,-0.113827,0.239101,-0.258062,-0.262159,-0.835531,-0.149293,-0.323698,-0.085351,-0.678723,-0.881682,-4.165102e-03,-0.175570
3,-0.522784,6.189614e-01,-0.094869,-0.060487,-0.467036,-0.173384,-2.199856,-0.127355,-0.290570,-0.079952,-0.587788,-0.935455,-1.002865e+00,-0.204763
4,-1.027965,1.883888e+00,-0.161224,-0.210281,-0.989470,-0.276363,-3.091915,2.573771,-0.191184,-0.073653,-0.618099,0.064728,-5.035150e-01,-0.058796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1276,-0.971834,5.049352e-16,0.000000,0.000000,0.000000,0.053881,0.345135,0.000000,-0.187504,-0.039458,0.806547,-0.537533,-9.147597e-10,0.000000
1277,-0.635046,5.049352e-16,0.000000,0.000000,0.000000,0.050330,0.738690,0.000000,0.217399,-0.009762,0.624678,-0.161120,-9.147597e-10,0.000000
1278,0.375317,5.049352e-16,0.000000,0.000000,0.000000,-0.084609,0.240186,0.000000,0.106971,-0.025060,0.382184,1.064910,-9.147597e-10,0.000000
1279,-0.388068,5.049352e-16,0.000000,0.000000,0.000000,0.014819,0.266424,0.000000,0.033352,-0.037658,0.109380,-0.602061,-9.147597e-10,0.000000


In [64]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [65]:
# Initialize the model
LG_model = LogisticRegression(random_state=42)

# Fit the model on the training data
LG_model.fit(X_train, y_train)

# Predict on the testing data
y_pred = LG_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.7587548638132295
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.86      0.87        81
           1       0.84      0.79      0.82        34
           2       1.00      0.40      0.57         5
           3       1.00      0.33      0.50         3
           4       0.00      0.00      0.00         1
           5       0.77      0.71      0.74        56
           6       0.59      0.85      0.70        46
           7       0.29      0.14      0.19        14
           8       0.82      0.82      0.82        17

    accuracy                           0.76       257
   macro avg       0.69      0.55      0.58       257
weighted avg       0.76      0.76      0.75       257



e:\AI\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning:

Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

e:\AI\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning:

Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

e:\AI\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning:

Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



In [66]:
# Decoding encoded classes
decoded_classes = label_encoder.inverse_transform(range(len(label_encoder.classes_)))
print("Original Anemia Types:", decoded_classes)

Original Anemia Types: ['Healthy' 'Iron deficiency anemia' 'Leukemia'
 'Leukemia with thrombocytopenia' 'Macrocytic anemia'
 'Normocytic hypochromic anemia' 'Normocytic normochromic anemia'
 'Other microcytic anemia' 'Thrombocytopenia']


In [67]:
category_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
category_mapping

{'Healthy': 0,
 'Iron deficiency anemia': 1,
 'Leukemia': 2,
 'Leukemia with thrombocytopenia': 3,
 'Macrocytic anemia': 4,
 'Normocytic hypochromic anemia': 5,
 'Normocytic normochromic anemia': 6,
 'Other microcytic anemia': 7,
 'Thrombocytopenia': 8}

In [68]:
fig = go.Figure(data=[go.Pie(labels=data['Diagnosis'], pull=[0.4,0],hole=0.5)])
fig.show()

In [69]:
# Apply SMOTE with reduced number of neighbors
smote = SMOTE(sampling_strategy='auto', k_neighbors=1, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [70]:
fig = go.Figure(data=[go.Pie(labels=y_train_resampled,hole=0.5)])
fig.show()

In [71]:
# Initialize the model
LG_model = LogisticRegression(random_state=42)

# Fit the model on the training data
LG_model.fit(X_train_resampled, y_train_resampled)

# Predict on the testing data
y_pred = LG_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.7587548638132295
Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.77      0.83        81
           1       0.88      0.88      0.88        34
           2       0.62      1.00      0.77         5
           3       0.33      1.00      0.50         3
           4       0.00      0.00      0.00         1
           5       0.83      0.62      0.71        56
           6       0.84      0.70      0.76        46
           7       0.54      1.00      0.70        14
           8       0.58      0.82      0.68        17

    accuracy                           0.76       257
   macro avg       0.62      0.75      0.65       257
weighted avg       0.82      0.76      0.78       257



e:\AI\Anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



In [72]:
# Initialize the model
DT_model = DecisionTreeClassifier(random_state=42)

# Fit the model on the resampled training data
DT_model.fit(X_train_resampled, y_train_resampled)

# Predict on the testing data
y_pred = DT_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9883268482490273
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        81
           1       1.00      0.97      0.99        34
           2       1.00      1.00      1.00         5
           3       0.60      1.00      0.75         3
           4       1.00      1.00      1.00         1
           5       1.00      1.00      1.00        56
           6       1.00      0.98      0.99        46
           7       0.93      0.93      0.93        14
           8       1.00      1.00      1.00        17

    accuracy                           0.99       257
   macro avg       0.95      0.99      0.96       257
weighted avg       0.99      0.99      0.99       257



In [76]:
from sklearn.model_selection import GridSearchCV

# Define hyperparameters to tune
params = {
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10,12],
    'min_samples_leaf': [1, 2, 3,4,5,6]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(DT_model, params, cv=10, scoring='accuracy')

# Fit GridSearchCV to the resampled training data
grid_search.fit(X_train_resampled, y_train_resampled)

# Get the best parameters
best_params = grid_search.best_params_
print("Best Hyperparameters:", best_params)


Best Hyperparameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [79]:
DT_model = DecisionTreeClassifier(random_state=42,max_depth=None,min_samples_split=2,min_samples_leaf=1)
DT_model.fit(X_train_resampled, y_train_resampled)

DecisionTreeClassifier(random_state=42)

In [80]:
# Predict on the testing data
y_pred = DT_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9883268482490273
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        81
           1       1.00      0.97      0.99        34
           2       1.00      1.00      1.00         5
           3       0.60      1.00      0.75         3
           4       1.00      1.00      1.00         1
           5       1.00      1.00      1.00        56
           6       1.00      0.98      0.99        46
           7       0.93      0.93      0.93        14
           8       1.00      1.00      1.00        17

    accuracy                           0.99       257
   macro avg       0.95      0.99      0.96       257
weighted avg       0.99      0.99      0.99       257



In [81]:
from sklearn.model_selection import cross_val_score

# Perform cross-validation
cv_scores = cross_val_score(grid_search.best_estimator_, X_train_resampled, y_train_resampled, cv=10)
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Accuracy:", np.mean(cv_scores))


Cross-Validation Scores: [1.         0.99565217 0.99565217 1.         1.         0.99563319
 1.         1.         0.99563319 1.        ]
Mean CV Accuracy: 0.998257072337194


In [86]:
import joblib

# Serialize the trained model
joblib.dump(LG_model, 'LG_model.pkl')
#joblib.dump(DT_model, 'DT_model.pkl')


['LG_model.pkl']

In [85]:
from flask import Flask, request, jsonify
import joblib

app = Flask(__name__)

# Load the serialized model
model = joblib.load('DT_model.pkl')

# Define mapping between encoded values and anemia names
anemia_mapping = {
    0: 'Healthy',
    1: 'Iron deficiency anemia',
    2: 'Leukemia',
    3: 'Leukemia with thrombocytopenia',
    4: 'Macrocytic anemia',
    5: 'Normocytic hypochromic anemia',
    6: 'Normocytic normochromic anemia',
    7: 'Other microcytic anemia',
    8: 'Thrombocytopenia'
    # Add mappings for other encoded values as needed
}

@app.route('/predict', methods=['POST'])
def predict():
# Get the input data from the request
    input_data = request.json
    print("Input data:", input_data)
    
    # Extract numeric values from the input data
    input_values = [value for value in input_data.values()]
    print("Input values:", input_values)
    
    # Perform prediction
    encoded_prediction = model.predict([input_values])[0]
    print("Encoded prediction:", encoded_prediction)
    
    # Map the encoded prediction to the corresponding anemia name
    predicted_anemia = anemia_mapping.get(encoded_prediction, 'Unknown')
    print("Predicted anemia:", predicted_anemia)
    
    # Return the predicted anemia name
    return jsonify({'predicted_anemia': predicted_anemia})

if __name__ == '__main__':
    app.run(debug=False, port=8080)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:8080
Press CTRL+C to quit


In [84]:
from flask import Flask, request, jsonify
import joblib

app = Flask(__name__)

# Load the serialized model
model = joblib.load('DT_model.pkl')

# Define mapping between encoded values and anemia names
anemia_mapping = {
    0: 'Healthy',
    1: 'Iron deficiency anemia',
    2: 'Leukemia',
    3: 'Leukemia with thrombocytopenia',
    4: 'Macrocytic anemia',
    5: 'Normocytic hypochromic anemia',
    6: 'Normocytic normochromic anemia',
    7: 'Other microcytic anemia',
    8: 'Thrombocytopenia'
    # Add mappings for other encoded values as needed
}

@app.route('/predict', methods=['POST'])
def predict():
    # Get the input data from the request
    input_data = request.json
    
    # Extract numeric values from the input data
    input_values = [value for value in input_data.values()]
    
    # Perform prediction
    encoded_prediction = model.predict([input_values])[0]
    
    # Map the encoded prediction to the corresponding anemia name
    predicted_anemia = anemia_mapping.get(encoded_prediction, 'Unknown')
    
    # Return the predicted anemia name
    return jsonify({'predicted_anemia': predicted_anemia})

if __name__ == '__main__':
    app.run(debug=True, port=8080)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:8080
Press CTRL+C to quit
